
# Importações

In [1]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error as mape, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

from pyESN import ESN
from shap.plots import colors
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from wsb import WSB

MODELOS = ["ESN", "MLP", "RF", "XGBoost", "WSB"]
SEED = 100
SEEDS = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
HORIZONTES = [3, 6, 12]
HORIZONTE = 12


def reset_seed(rnd_seed=SEED):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)
    np.random.seed(rnd_seed)


def calcular_rrmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    mean_y_true = np.mean(y_true)

    rrmse = rmse / mean_y_true
    return rrmse


warnings.filterwarnings("ignore")
reset_seed()

C:\Users\eduar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar Datasets

In [2]:
df = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

## Normalização

In [3]:
# Repete a normalização para obtermos so scalers correspondentes
scalers = {}
dataframes = []

for campus, dados in df.groupby("CAMPUS"):
    scaler = MinMaxScaler()
    dados[["CONSUMO"]] = scaler.fit_transform(dados[["CONSUMO"]])

    scalers[campus] = scaler
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)

## Criação dos Lags

In [4]:
dataframes = []

for campus, dados in df.sort_values("DATA").groupby("CAMPUS"):
    lags = {f'LAG_{i:02d}': dados['CONSUMO'].shift(i) for i in range(1, 12 + 1)}
    dados = pd.concat([dados, pd.DataFrame(lags)], axis=1)
    dados.dropna(inplace=True)
    dados["ORDEM"] = range(1, len(dados) + 1)
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)
df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
0,0.615105,2016-02-29,19,22,25,28,734,33,4,19,...,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886,0.502037
1,0.711047,2016-03-31,12,17,22,28,685,32,3,12,...,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886
2,0.633361,2016-04-30,4,9,23,28,702,33,2,4,...,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078
3,0.406291,2016-05-31,4,11,16,22,490,27,9,4,...,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844
4,0.362467,2016-06-30,-1,7,14,20,408,27,3,-1,...,0.711047,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248,0.743880
2091,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248
2092,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038
2093,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,0.359224,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059


## Melhores Features

In [5]:
df_features = pd.read_csv("./resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")

df_features = df_features.sort_values("RRMSE").head(1).reset_index(drop=True)
df_features = pd.DataFrame(
    columns=str(df_features.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "))

df_features = df_features.columns

df_features

Index(['TEMP_MÉD_MIN_MENS', 'TEMP_MÉD_MÉD_MENS', 'PRECIPITAÇÃO_MÉD_MENS',
       'TEMP_MIN_MAX_MENS', 'TEMP_MAX_MIN_MENS', 'PRECIPITAÇÃO_MIN_MENS',
       'TEMP_MAX_MAX_MENS', 'DIA_DA_SEMANA_dom', 'DIA_DA_SEMANA_seg',
       'DIA_DA_SEMANA_sáb', 'DIA_DA_SEMANA_ter', 'MÊS_abr', 'MÊS_ago',
       'MÊS_fev', 'MÊS_jun', 'MÊS_mai', 'MÊS_nov', 'ANO_2021', 'ANO_2022',
       'ANO_2023', 'ANO_2015', 'ANO_2016', 'ANO_2017', 'ANO_2018', 'ANO_2019',
       'CAMPUS_ASTORGA', 'CAMPUS_CAMPO LARGO', 'CAMPUS_CAPANEMA',
       'CAMPUS_CASCAVEL', 'CAMPUS_CORONEL VIVIDA', 'CAMPUS_CURITIBA',
       'CAMPUS_GOIOERÊ', 'CAMPUS_IVAIPORÃ', 'CAMPUS_JAGUARIAÍVA',
       'CAMPUS_LONDRINA - CENTRO', 'CAMPUS_PALMAS', 'CAMPUS_PARANAGUÁ',
       'CAMPUS_PINHAIS', 'CAMPUS_TELÊMACO BORBA', 'CAMPUS_UMUARAMA',
       'CURSOS_TEC_SUBSEQUENTE', 'CURSOS_GRAD_MATUTINO',
       'CURSOS_GRAD_VESPERTINO', 'CURSOS_GRAD_NOTURNO', 'CURSOS_POS', 'FÉRIAS',
       'COVID', 'LAG_01', 'LAG_02', 'LAG_03', 'LAG_05', 'LAG_07', 'LAG_09'],


## Melhores Parâmetros

In [6]:
def get_modelo(nome, tipo_treino=None, campus=None):
    if nome == "ESN":
        return ESN(n_inputs=df_features.shape[0],
                   n_outputs=1,
                   n_reservoir=int(best["ESN"]["Reservoirs"]),
                   sparsity=best["ESN"]["Sparsity"],
                   spectral_radius=best["ESN"]["Spectral Radius"],
                   random_state=int(best["ESN"]["SEED"]))

    if nome == "MLP":
        mlp = MLPRegressor(hidden_layer_sizes=(int(best["MLP"]["Hidden Layers"]),),
                           activation=best["MLP"]["Activation"],
                           alpha=best["MLP"]["Alpha"],
                           random_state=int(best["MLP"]["SEED"]))
        return mlp

    if nome == "RF":
        return RandomForestRegressor(random_state=int(best["RF"]["SEED"]),
                                     n_estimators=int(best["RF"]["N_estimators"]),
                                     max_depth=int(best["RF"]["Max_depth"]),
                                     min_samples_split=int(best["RF"]["Min_samples_split"]),
                                     min_samples_leaf=int(best["RF"]["Min_samples_leaf"]))

    if nome == "XGBoost":
        return XGBRegressor(random_state=int(best["XGBoost"]["SEED"]),
                            n_estimators=int(best["XGBoost"]["N_estimators"]),
                            max_depth=int(best["XGBoost"]["Max_depth"]),
                            booster=best["XGBoost"]["Booster"],
                            reg_lambda=best["XGBoost"]["Lambda"],
                            reg_alpha=best["XGBoost"]["Alpha"],
                            updater="coord_descent" if best["XGBoost"]["Booster"] == "gblinear" else None)

    if nome == "WSB":
        if tipo_treino is None or campus is None:
            raise ValueError("Para WSB, 'tipo_treino' e 'campus' devem ser fornecidos.")
        return WSB(strong_predictor=get_modelo(wsb_prev_forte[(campus, tipo_treino)][0]),
                   weak_predictors=[get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred],
                   weight_g=wsb_prev_forte[(campus, tipo_treino)][1])


wsb_prev_forte = {}
best = {}
for modelo in MODELOS:
    if modelo == "WSB":
        continue
    df_aux = pd.read_csv(
        f"./resultados/otimização - regressão/BEST {modelo}.csv", sep=';',
        decimal='.', header=0)

    best[modelo] = df_aux.iloc[0]

for key, val in best.items():
    display(val)

OTIMIZADOR           PSO
MODELO               ESN
SEED                9000
Reservoirs          11.0
Sparsity            0.23
Spectral Radius    0.639
Fitness            0.289
Name: 0, dtype: object

OTIMIZADOR         PSO
MODELO             MLP
SEED             10000
Hidden Layers      220
Alpha            0.979
Activation        relu
Fitness          0.227
Name: 0, dtype: object

OTIMIZADOR             PSO
MODELO                  RF
SEED                  7000
N_estimators          15.0
Max_depth            232.0
Min_samples_split     11.0
Min_samples_leaf       4.0
Fitness              0.247
Name: 0, dtype: object

OTIMIZADOR          PSO
MODELO          XGBoost
SEED               5000
N_estimators        242
Max_depth           119
Booster          gbtree
Lambda            0.898
Alpha             0.041
Fitness           0.243
Name: 0, dtype: object

# Divisão dos Dados


In [7]:
df_treino = []
df_teste = []

for campus, dados in df.sort_values('DATA').groupby("CAMPUS"):
    dados["CAMPUS"] = campus

    dados_treino, dados_teste = train_test_split(dados, test_size=HORIZONTE, shuffle=False)

    df_treino.append(dados_treino)
    df_teste.append(dados_teste)

df_treino = pd.DataFrame(pd.concat(df_treino, ignore_index=True))
df_teste = pd.DataFrame(pd.concat(df_teste, ignore_index=True))


# Execução dos Experimentos

In [8]:
def treino(previsor, dados_treino, features):
    x_treino = dados_treino[features].to_numpy()
    y_treino = dados_treino["CONSUMO"].to_numpy()

    previsor.fit(x_treino, y_treino)
    return previsor


def teste(previsor, historico_campus, x_teste, features, horizonte=HORIZONTE):
    historico = historico_campus[["CONSUMO"]].copy()
    x_teste = x_teste[features].copy()

    previsoes = []

    for i_test in range(horizonte):
        row = x_teste.iloc[[i_test]].copy()
        historico = pd.concat([historico, pd.DataFrame([0], columns=["CONSUMO"], index=[i_test])], axis=0)

        # Recalcula os lags conforme os valores previstos pelo modelo
        lags = pd.DataFrame({f'LAG_{i:02d}': historico["CONSUMO"].shift(i) for i in range(1, 12 + 1) if
                             f'LAG_{i:02d}' in features}).tail(1)
        row.update(lags)

        if isinstance(previsor, WSB) and horizonte >= 6:
            peso_t = i_test / horizonte
            prev = previsor.predict(row.to_numpy(), peso_t)[0]
        elif isinstance(previsor, ESN):
            prev = previsor.predict(row.to_numpy())[0][0]
        else:
            prev = previsor.predict(row.to_numpy())[0]

        row["CONSUMO"] = prev
        previsoes.append(prev)
        historico.update(row)

    return pd.DataFrame({"CONSUMO PREVISTO": previsoes}, index=x_teste.index)

## Definição dos Previsores Fortes - Local

In [ ]:
# Define o previsor forte para o WSB com base no menor RRMSE histórico por campus
# Treinamento local
for campus, dados in df_treino.sort_values("DATA").groupby("CAMPUS"):
    dados = dados.set_index('DATA')
    dados_treino, dados_teste = train_test_split(dados, test_size=12, shuffle=False)

    erros = []
    for nome_modelo in MODELOS:
        if nome_modelo == "WSB":
            continue
        # Treina o modelo com os dados do campus atual
        strong_pred = get_modelo(nome_modelo)
        weak_preds = [get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred]
        modelo = WSB(strong_predictor=strong_pred, weak_predictors=weak_preds, weight_g=-1)
        modelo = treino(modelo, dados, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, dados, dados_teste, df_features, 12)
        erro = (dados_teste["CONSUMO"].mean() - previsoes["CONSUMO PREVISTO"].mean()) / dados_teste["CONSUMO"].mean()
        rrmse = calcular_rrmse(dados_teste["CONSUMO"].to_numpy(), previsoes["CONSUMO PREVISTO"].to_numpy())
        erros.append([rrmse, erro, nome_modelo])

    # Define o modelo com menor RRMSE como o previsor forte
    erros = sorted(erros, key=lambda x: np.abs(x[0]))
    wsb_prev_forte[(campus, "local")] = [erros[0][2], -1 + erros[0][1]]

# Converte o dicionário wsb_prev_forte em um DataFrame
df_wsb_prev_forte = pd.DataFrame.from_dict(wsb_prev_forte, orient="index", columns=["MODELO", "WEIGHT_G"])
df_wsb_prev_forte.index = pd.MultiIndex.from_tuples(df_wsb_prev_forte.index, names=["CAMPUS", "TIPO_TREINAMENTO"])

# Salva o DataFrame em um arquivo CSV
df_wsb_prev_forte.to_csv(f"./resultados/WSB - Previsores Fortes {HORIZONTE}M.csv", sep=';', decimal='.')
df_wsb_prev_forte

## Definição dos Previsores Fortes - Global

In [10]:
# Treinamento global
df_aux_treino = []
df_aux_test = []

for campus, dados in df_treino.sort_values("DATA").groupby("CAMPUS"):
    dados_treino, dados_teste = train_test_split(dados, test_size=12, shuffle=False)
    df_aux_treino.append(dados_treino)
    df_aux_test.append(dados_teste)

df_aux_treino = pd.DataFrame(pd.concat(df_aux_treino, ignore_index=True))
df_aux_test = pd.DataFrame(pd.concat(df_aux_test, ignore_index=True))

for campus, dados in df_aux_test.sort_values("DATA").groupby("CAMPUS"):
    dados = dados.set_index('DATA')

    erros = []
    for nome_modelo in MODELOS:
        if nome_modelo == "WSB":
            continue
        # Treina o modelo com os dados de todos os campi
        strong_pred = get_modelo(nome_modelo)
        weak_preds = [get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred]
        modelo = WSB(strong_predictor=strong_pred, weak_predictors=weak_preds, weight_g=-1)
        modelo = treino(modelo, df_aux_treino, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, df_aux_treino[df_aux_treino["CAMPUS"] == campus], dados, df_features, 12)
        erro = (dados["CONSUMO"].mean() - previsoes["CONSUMO PREVISTO"].mean()) / dados["CONSUMO"].mean()
        rrmse = calcular_rrmse(dados["CONSUMO"].to_numpy(), previsoes["CONSUMO PREVISTO"].to_numpy())
        erros.append([rrmse, erro, nome_modelo])

    # Define o modelo com menor RRMSE como o previsor forte
    erros = sorted(erros, key=lambda x: np.abs(x[0]))
    wsb_prev_forte[(campus, "global")] = [erros[0][2], -1 + erros[0][1]]

# Converte o dicionário wsb_prev_forte em um DataFrame
df_wsb_prev_forte = pd.DataFrame.from_dict(wsb_prev_forte, orient="index", columns=["MODELO", "WEIGHT_G"])
df_wsb_prev_forte.index = pd.MultiIndex.from_tuples(df_wsb_prev_forte.index, names=["CAMPUS", "TIPO_TREINAMENTO"])

# Salva o DataFrame em um arquivo CSV
df_wsb_prev_forte.to_csv(f"./resultados/WSB - Previsores Fortes {HORIZONTE}M.csv", sep=';', decimal='.')
df_wsb_prev_forte



## Treinamento Local

In [11]:

for campus, dados_teste in df_teste.sort_values("DATA").groupby("CAMPUS"):
    dados_teste = dados_teste.set_index('DATA')
    df_previsoes = pd.DataFrame(columns=MODELOS, index=dados_teste.index)

    for nome_modelo in MODELOS:
        # Treina o modelo com os dados do campus atual
        dados_treino = df_treino[df_treino["CAMPUS"] == campus]
        modelo = treino(get_modelo(nome_modelo, tipo_treino="local", campus=campus), dados_treino, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, dados_treino, dados_teste, df_features)
        df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

    os.makedirs(f"resultados/regressão - local/{HORIZONTE} meses", exist_ok=True)
    df_previsoes.to_csv(f"resultados/regressão - local/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M - {campus}.csv",
                        sep=";", decimal=".", header=True,
                        index=True)


## Treinamento Global

In [12]:

for campus, dados_teste in df_teste.sort_values("DATA").groupby("CAMPUS"):
    dados_teste = dados_teste.set_index('DATA')
    df_previsoes = pd.DataFrame(columns=MODELOS, index=dados_teste.index)

    for nome_modelo in MODELOS:
        # Treina o modelo com os dados de todos os campi
        modelo = treino(get_modelo(nome_modelo, tipo_treino="global", campus=campus), df_treino, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, df_treino[df_treino["CAMPUS"] == campus], dados_teste, df_features)
        df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

    os.makedirs(f"resultados/regressão - global/{HORIZONTE} meses", exist_ok=True)
    df_previsoes.to_csv(f"resultados/regressão - global/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M - {campus}.csv",
                        sep=";", decimal=".",
                        index=True)



# Análise dos Resultados

In [13]:
def ts_comparacao(campus, valor_real, valores_previstos, erros):
    valor_real = valor_real.tail(HORIZONTE)
    plt.figure(figsize=(12, 4.5))
    plt.rcParams['xtick.labelsize'] = 13
    plt.rcParams['ytick.labelsize'] = 14
    plt.rcParams.update({'font.size': 12})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", "green", "darkgoldenrod", colors.red_rgb, "purple", "cyan", "slategrey", "coral"])

    for nome_modelo in valores_previstos.columns:
        plt.plot(valores_previstos[nome_modelo],
                 label=f"{nome_modelo} (RRMSE: {erros.loc[nome_modelo]["RRMSE"]:.2%} - MAPE: {erros.loc[nome_modelo]["MAPE"]:.2%})")

    plt.plot(valor_real["CONSUMO"], label=f"CONSUMO REAL - {campus}", color="black")

    plt.xlabel('Mês')
    plt.ylabel('Consumo (KWh)')

    ax = plt.gca()
    ax.set_facecolor('white')

    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')

    return plt


def medidas_desempenho(valor_real, valores_previstos):
    df_desempenho = pd.DataFrame(columns=["MAPE", "RRMSE"], index=valores_previstos.columns)

    for nome_modelo in valores_previstos.columns:
        df_desempenho.loc[nome_modelo] = [
            mape(valor_real["CONSUMO"].tail(HORIZONTE), valores_previstos[nome_modelo].tail(HORIZONTE)),
            calcular_rrmse(valor_real["CONSUMO"].tail(HORIZONTE), valores_previstos[nome_modelo].tail(HORIZONTE)),
        ]

    return df_desempenho


## Treinamento Local

In [14]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

df_RRMSE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())
df_MAPE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())

for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
    try:
        consumo_previsto = pd.read_csv(
            f"resultados/regressão - local/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M - {campus}.csv",
            sep=";", decimal=".", header=0)
    except Exception as e:
        continue

    consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
    consumo_previsto = consumo_previsto.set_index("DATA")

    dados["DATA"] = pd.to_datetime(dados["DATA"])
    dados = dados.set_index("DATA")

    df_desempenho = medidas_desempenho(dados, consumo_previsto)
    df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
    df_MAPE.loc[campus] = df_desempenho["MAPE"]

    plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho)

    df_desempenho.to_csv(f"resultados/regressão - local/{HORIZONTE} meses/RRMSE {HORIZONTE}M {campus}.csv",
                         sep=";", decimal=".", index=True)
    plt.savefig(f"resultados/regressão - local/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M {campus}.png",
                bbox_inches='tight')
    plt.close()

df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
df_MAPE = df_MAPE.add_suffix(" MAPE")

pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))

df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
df_medias.loc["MÉDIAS"] = df_medias.mean()

df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

df_medias.to_csv(f"resultados/regressão - local/MÉDIAS ERROS {HORIZONTE}M.csv", sep=";", decimal=".", index=True)
df_medias



,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.280578,0.398155,0.36148,0.345205,0.344896,0.178217,0.551513,0.41961,0.41411,0.419487
ASTORGA,0.477217,0.308458,0.306776,0.334848,0.327267,0.430907,0.236664,0.228922,0.255418,0.245007
BARRACÃO,0.288687,0.229966,0.295104,0.27954,0.273175,0.233853,0.21752,0.26805,0.242562,0.240372
CAMPO LARGO,0.365603,0.334673,0.335989,0.326037,0.311579,0.358701,0.378977,0.355507,0.320603,0.328726
CAPANEMA,0.535447,0.431023,0.395956,0.340894,0.364174,0.470756,0.550737,0.436529,0.365215,0.396548
CASCAVEL,0.273705,0.298589,0.326426,0.283239,0.273274,0.283747,0.285663,0.328,0.256836,0.256368
CORONEL VIVIDA,0.345348,0.34397,0.267654,0.239236,0.254817,0.304481,0.379106,0.287886,0.250438,0.268652
CURITIBA,0.841661,0.393413,0.295968,0.354607,0.31337,0.668006,0.250875,0.264717,0.309176,0.260667
FOZ DO IGUAÇU,0.385475,0.512756,0.438348,0.485188,0.476748,0.423062,0.566678,0.510157,0.518693,0.52358
GOIOERÊ,0.25546,0.266759,0.304176,0.277809,0.269538,0.272904,0.298071,0.276366,0.281362,0.262461


## Treinamento Global

In [15]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

df_RRMSE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())
df_MAPE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())

for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
    try:
        consumo_previsto = pd.read_csv(
            f"resultados/regressão - global/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M - {campus}.csv",
            sep=";", decimal=".", header=0)
    except Exception as e:
        continue

    consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
    consumo_previsto = consumo_previsto.set_index("DATA")

    dados["DATA"] = pd.to_datetime(dados["DATA"])
    dados = dados.set_index("DATA")

    df_desempenho = medidas_desempenho(dados, consumo_previsto)
    df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
    df_MAPE.loc[campus] = df_desempenho["MAPE"]

    plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho)

    df_desempenho.to_csv(f"resultados/regressão - global/{HORIZONTE} meses/RRMSE {HORIZONTE}M {campus}.csv",
                         sep=";", decimal=".", index=True)
    plt.savefig(f"resultados/regressão - global/{HORIZONTE} meses/PREVISÕES {HORIZONTE}M {campus}.png",
                bbox_inches='tight')
    plt.close()


df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
df_MAPE = df_MAPE.add_suffix(" MAPE")

pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))


df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
df_medias.loc["MÉDIAS"] = df_medias.mean()

df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

df_medias.to_csv(f"resultados/regressão - global/MÉDIAS ERROS {HORIZONTE}M.csv", sep=";", decimal=".", index=True)
df_medias

